# DCGAN vs LSGAN vs WGAN-GP — CelebA Faces
64x64 human faces, subsampled to ~30k images. Face generation is a listed project direction. Each dataset is trained on **three GAN objectives — DCGAN, LSGAN, WGAN-GP** — then compared on FID / Inception Score, and the winner is promoted for deployment.

WGAN-GP runs ~`N_CRITIC`× more discriminator steps and trains in fp32, so it is the slow one. Training is checkpoint-resumable per architecture, so a disconnect just resumes. To fit a short session, drop an entry from `ARCHITECTURES` and rerun later.

Estimated time on T4: 8-14 hours for all three (resumable across sessions).

## Install dependencies

In [ ]:
!pip install -q torchmetrics[image] kagglehub

## Environment & paths (Colab Drive checkpoint / Kaggle / local)

In [ ]:
import os


def detect_env():
    if os.path.exists("/kaggle/working"):
        return "kaggle"
    if os.path.exists("/content"):
        return "colab"
    return "local"


ENV = detect_env()

if ENV == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_DIR = "/content/drive/MyDrive/image-lab-checkpoints"
    WEIGHTS_DIR = "/content/weights"
    SAMPLES_DIR = "/content/samples"
elif ENV == "kaggle":
    CKPT_DIR = "/kaggle/working/checkpoints"
    WEIGHTS_DIR = "/kaggle/working/weights"
    SAMPLES_DIR = "/kaggle/working/samples"
else:
    CKPT_DIR = "checkpoints"
    WEIGHTS_DIR = "weights"
    SAMPLES_DIR = "samples"

for d in (CKPT_DIR, WEIGHTS_DIR, SAMPLES_DIR):
    os.makedirs(d, exist_ok=True)

PREFIX = "celeba"
print("environment:", ENV, "| checkpoints ->", CKPT_DIR)

## Imports

In [ ]:
import glob
import random

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torchvision.utils import make_grid
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Hyperparameters

In [ ]:
NZ = 100
IMG_SIZE = 64
CHANNELS = 3
BATCH_SIZE = 128
LR = 2e-4
BETAS = (0.5, 0.999)

# ---- comparison / training budget ----
ARCHITECTURES = ["dcgan", "lsgan", "wgan_gp"]   # trim to fit a short GPU session
EPOCHS = 60
CKPT_EVERY = 5        # sample + FID + checkpoint cadence
FID_SAMPLES = 2048    # samples per FID/IS estimate
N_CRITIC = 5          # WGAN-GP critic steps per generator step (lower = faster)

## Locate / download CelebA

In [ ]:
import kagglehub

def find_image_folder(root):
    best, best_n = None, 0
    for dp, _, files in os.walk(root):
        n = sum(1 for f in files if f.lower().endswith((".jpg", ".jpeg", ".png")))
        if n > best_n:
            best, best_n = dp, n
    return best, best_n

img_root = None
if ENV == "kaggle" and os.path.exists("/kaggle/input"):
    img_root, n = find_image_folder("/kaggle/input")

if not img_root:
    try:
        dataset_path = kagglehub.dataset_download("jessicali9530/celeba-dataset")
    except Exception:
        kagglehub.login()
        dataset_path = kagglehub.dataset_download("jessicali9530/celeba-dataset")
    img_root, n = find_image_folder(dataset_path)

print(f"image folder: {img_root}  ({n} images)")
assert img_root, "no CelebA images found"

## Subsample & dataset

In [ ]:
random.seed(42)
N_SUBSET = 30000

all_files = sorted(glob.glob(os.path.join(img_root, "*.jpg")) + glob.glob(os.path.join(img_root, "*.png")))
random.shuffle(all_files)
subset_files = all_files[:N_SUBSET]
print(f"using {len(subset_files)} of {len(all_files)} images")


class ImageFolderFlat(Dataset):
    def __init__(self, files, transform):
        self.files, self.transform = files, transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        return self.transform(Image.open(self.files[idx]).convert("RGB")), 0


transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

dataset = ImageFolderFlat(subset_files, transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)

## Models — Generator, Discriminator (DCGAN/LSGAN), Critic (WGAN-GP)

In [ ]:
import math
import torch.nn as nn


def _gen_config(img_size):
    for init in (4, 7):
        if img_size % init == 0:
            n = img_size // init
            steps = int(math.log2(n))
            if init * (2 ** steps) == img_size:
                return init, steps
    raise ValueError(f"unsupported img_size {img_size}")


class Generator(nn.Module):
    """Latent vector -> image. Self-configures depth from img_size. Shared by all
    three GAN variants. img_size is required."""

    def __init__(self, img_size, nz=100, ngf=64, channels=3):
        super().__init__()
        init_size, steps = _gen_config(img_size)
        cur = ngf * 2 ** max(steps - 1, 0)
        layers = [nn.ConvTranspose2d(nz, cur, init_size, 1, 0, bias=False),
                  nn.BatchNorm2d(cur), nn.ReLU(True)]
        for i in range(steps):
            last = i == steps - 1
            nxt = channels if last else cur // 2
            layers.append(nn.ConvTranspose2d(cur, nxt, 4, 2, 1, bias=False))
            layers += [nn.Tanh()] if last else [nn.BatchNorm2d(nxt), nn.ReLU(True)]
            cur = nxt
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        return self.net(z)


def _disc_body(img_size, channels, norm):
    """Conv down-sampling stack shared by the DCGAN/LSGAN discriminator and the
    WGAN-GP critic. `norm` is the normalization layer (BatchNorm for DCGAN/LSGAN,
    InstanceNorm for WGAN-GP — BatchNorm breaks the gradient penalty)."""
    init_size, steps = _gen_config(img_size)
    layers = []
    cur, nxt = channels, 64
    for i in range(steps):
        layers.append(nn.Conv2d(cur, nxt, 4, 2, 1, bias=False))
        if i > 0:
            layers.append(norm(nxt))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        cur = nxt
        if i < steps - 1:
            nxt *= 2
    layers.append(nn.Conv2d(cur, 1, init_size, 1, 0, bias=False))
    return nn.Sequential(*layers)


class Discriminator(nn.Module):
    """DCGAN / LSGAN. Outputs a raw score (no Sigmoid) — DCGAN pairs it with
    BCEWithLogitsLoss (autocast-safe), LSGAN with MSELoss."""

    def __init__(self, img_size, channels=3):
        super().__init__()
        self.net = _disc_body(img_size, channels, nn.BatchNorm2d)

    def forward(self, x):
        return self.net(x).view(-1)


class Critic(nn.Module):
    """WGAN-GP critic. InstanceNorm instead of BatchNorm so the gradient penalty
    is well-defined; outputs an unbounded score."""

    def __init__(self, img_size, channels=3):
        super().__init__()
        self.net = _disc_body(img_size, channels, lambda c: nn.InstanceNorm2d(c, affine=True))

    def forward(self, x):
        return self.net(x).view(-1)


def weights_init(m):
    cn = m.__class__.__name__
    if "Conv" in cn:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "BatchNorm" in cn:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

## Mixed-precision setup (version-safe)

In [ ]:
# Version-safe AMP: new torch.amp API when available, else torch.cuda.amp.
try:
    from torch.amp import autocast as _autocast, GradScaler as _GradScaler
    _GradScaler("cuda")  # probe (raises on old torch)
    def make_scaler(enabled):
        return _GradScaler("cuda", enabled=enabled)
    def amp_autocast(enabled):
        return _autocast("cuda", enabled=enabled)
    print("AMP API: torch.amp")
except (ImportError, TypeError, RuntimeError):
    from torch.cuda.amp import autocast as _autocast, GradScaler as _GradScaler
    def make_scaler(enabled):
        return _GradScaler(enabled=enabled)
    def amp_autocast(enabled):
        return _autocast(enabled=enabled)
    print("AMP API: torch.cuda.amp")

## Display helper + real-sample preview

In [ ]:
CMAP = "gray" if CHANNELS == 1 else None


def show_grid(img_grid, title=None, save=None, figsize=(4, 4)):
    arr = img_grid.permute(1, 2, 0)
    if CHANNELS == 1:
        arr = arr.squeeze()
    plt.figure(figsize=figsize)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.imshow(arr, cmap=CMAP)
    if save:
        plt.savefig(save, bbox_inches="tight")
    plt.show()


real_batch, _ = next(iter(dataloader))
show_grid(make_grid(real_batch[:16], nrow=4, normalize=True), title="real samples")

## Evaluation metrics — FID & Inception Score

In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore


@torch.no_grad()
def compute_fid_is(gen, n_samples=2048, batch=128):
    """FID (lower better) and Inception Score (higher better) for a generator.
    Inception wants 3-channel [0,1] images; we un-normalize from [-1,1] and
    expand grayscale. FID on MNIST is unconventional but a valid relative signal."""
    fid = FrechetInceptionDistance(normalize=True).to(device)
    iscore = InceptionScore(normalize=True).to(device)
    gen.eval()

    def prep(x):
        x = (x.clamp(-1, 1) + 1) / 2
        return x.repeat(1, 3, 1, 1) if x.size(1) == 1 else x

    seen = 0
    for real, _ in dataloader:
        fid.update(prep(real.to(device)), real=True)
        seen += real.size(0)
        if seen >= n_samples:
            break
    seen = 0
    while seen < n_samples:
        z = torch.randn(batch, NZ, 1, 1, device=device)
        fake = prep(gen(z))
        fid.update(fake, real=False)
        iscore.update(fake)
        seen += batch
    gen.train()
    return fid.compute().item(), iscore.compute()[0].item()

## Training engine (checkpoint-resumable, one function per architecture)

In [ ]:
import json


def gradient_penalty(critic, real, fake):
    bs = real.size(0)
    eps = torch.rand(bs, 1, 1, 1, device=real.device)
    interp = (eps * real + (1 - eps) * fake).requires_grad_(True)
    scores = critic(interp)
    grads = torch.autograd.grad(
        outputs=scores, inputs=interp, grad_outputs=torch.ones_like(scores),
        create_graph=True, retain_graph=True, only_inputs=True)[0]
    grads = grads.reshape(bs, -1)
    return ((grads.norm(2, dim=1) - 1) ** 2).mean()


def build_models(arch):
    G = Generator(img_size=IMG_SIZE, nz=NZ, channels=CHANNELS).to(device)
    D = (Critic if arch == "wgan_gp" else Discriminator)(img_size=IMG_SIZE, channels=CHANNELS).to(device)
    G.apply(weights_init)
    D.apply(weights_init)
    return G, D


def train_gan(arch, epochs, checkpoint_every=5, fid_samples=2048, n_critic=5, gp_lambda=10.0):
    """Train one architecture end-to-end with checkpoint-resume. Saves per-arch
    final + best-FID generators, samples, and history. Returns the history dict."""
    ckpt_path = os.path.join(CKPT_DIR, f"{PREFIX}_{arch}_checkpoint.pth")
    best_path = os.path.join(WEIGHTS_DIR, f"{PREFIX}_{arch}_generator_best.pth")
    final_path = os.path.join(WEIGHTS_DIR, f"{PREFIX}_{arch}_generator.pth")
    hist_path = os.path.join(SAMPLES_DIR, f"{PREFIX}_{arch}_history.json")

    G, D = build_models(arch)
    betas = (0.5, 0.9) if arch == "wgan_gp" else BETAS
    optG = torch.optim.Adam(G.parameters(), lr=LR, betas=betas)
    optD = torch.optim.Adam(D.parameters(), lr=LR, betas=betas)
    use_amp = (device.type == "cuda") and (arch != "wgan_gp")  # GP needs fp32 double-backward
    scalerG, scalerD = make_scaler(use_amp), make_scaler(use_amp)
    fixed_noise = torch.randn(16, NZ, 1, 1, device=device)

    history = {"g_loss": [], "d_loss": [], "dx": [], "dgz": [], "fid": []}
    start_epoch, best_fid = 0, float("inf")
    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=device)
        G.load_state_dict(ck["G"]); D.load_state_dict(ck["D"])
        optG.load_state_dict(ck["optG"]); optD.load_state_dict(ck["optD"])
        start_epoch, best_fid, history = ck["epoch"] + 1, ck["best_fid"], ck["history"]
        fixed_noise = ck["fixed_noise"].to(device)
        print(f"[{arch}] resumed at epoch {start_epoch} (best FID {best_fid:.2f})")

    if start_epoch >= epochs:
        print(f"[{arch}] already complete — skipping")
        return history

    bce, mse = torch.nn.BCEWithLogitsLoss(), torch.nn.MSELoss()

    for epoch in range(start_epoch, epochs):
        gl = dl = dx = dgz = 0.0
        nb = 0
        for real, _ in dataloader:
            real = real.to(device)
            bs = real.size(0)

            if arch == "wgan_gp":
                for _ in range(n_critic):
                    fake = G(torch.randn(bs, NZ, 1, 1, device=device)).detach()
                    d_real, d_fake = D(real).mean(), D(fake).mean()
                    lossD = d_fake - d_real + gp_lambda * gradient_penalty(D, real.data, fake.data)
                    optD.zero_grad(); lossD.backward(); optD.step()
                fake = G(torch.randn(bs, NZ, 1, 1, device=device))
                lossG = -D(fake).mean()
                optG.zero_grad(); lossG.backward(); optG.step()
                dx += d_real.item(); dgz += d_fake.item()
            else:
                optD.zero_grad()
                with amp_autocast(use_amp):
                    fake = G(torch.randn(bs, NZ, 1, 1, device=device))
                    out_real, out_fake = D(real), D(fake.detach())
                    if arch == "dcgan":
                        lossD = bce(out_real, torch.full((bs,), 0.9, device=device)) + \
                                bce(out_fake, torch.zeros(bs, device=device))
                    else:  # lsgan
                        lossD = mse(out_real, torch.ones(bs, device=device)) + \
                                mse(out_fake, torch.zeros(bs, device=device))
                scalerD.scale(lossD).backward(); scalerD.step(optD); scalerD.update()

                optG.zero_grad()
                with amp_autocast(use_amp):
                    out = D(fake)
                    tgt = torch.ones(bs, device=device)
                    lossG = bce(out, tgt) if arch == "dcgan" else mse(out, tgt)
                scalerG.scale(lossG).backward(); scalerG.step(optG); scalerG.update()
                dx += torch.sigmoid(out_real.detach()).mean().item()
                dgz += torch.sigmoid(out_fake.detach()).mean().item()

            gl += lossG.item(); dl += lossD.item(); nb += 1

        history["g_loss"].append(gl / nb); history["d_loss"].append(dl / nb)
        history["dx"].append(dx / nb); history["dgz"].append(dgz / nb)
        extra = "" if arch == "wgan_gp" else f"  D(x)={dx / nb:.3f}  D(G(z))={dgz / nb:.3f}"
        print(f"[{arch}] epoch {epoch + 1}/{epochs}  loss_g={gl / nb:.3f}  loss_d={dl / nb:.3f}{extra}")

        if (epoch + 1) == 1 or (epoch + 1) % checkpoint_every == 0 or (epoch + 1) == epochs:
            G.eval()
            with torch.no_grad():
                samp = G(fixed_noise).cpu()
            G.train()
            show_grid(make_grid(samp, nrow=4, normalize=True), title=f"{arch} — epoch {epoch + 1}",
                      save=os.path.join(SAMPLES_DIR, f"{PREFIX}_{arch}_epoch{epoch + 1:03d}.png"))
            fid, isc = compute_fid_is(G, fid_samples)
            history["fid"].append((epoch + 1, fid, isc))
            print(f"    [{arch}] FID={fid:.2f}  IS={isc:.2f}")
            if fid < best_fid:
                best_fid = fid
                torch.save(G.state_dict(), best_path)
            torch.save({"epoch": epoch, "best_fid": best_fid, "history": history,
                        "G": G.state_dict(), "D": D.state_dict(),
                        "optG": optG.state_dict(), "optD": optD.state_dict(),
                        "fixed_noise": fixed_noise.cpu()}, ckpt_path)
            json.dump(history, open(hist_path, "w"))

    torch.save(G.state_dict(), final_path)
    print(f"[{arch}] done — final {final_path} | best (FID {best_fid:.2f}) {best_path}")
    return history

## Train all architectures

In [ ]:
histories = {}
for arch in ARCHITECTURES:
    print("=" * 60, f"\ntraining {arch}\n" + "=" * 60)
    histories[arch] = train_gan(arch, EPOCHS, checkpoint_every=CKPT_EVERY,
                                fid_samples=FID_SAMPLES, n_critic=N_CRITIC)

## Architecture comparison + pick the winner

In [ ]:
# ---- overlay curves ----
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
for arch in ARCHITECTURES:
    h = histories[arch]
    ax[0].plot(h["g_loss"], label=f"{arch} G")
    ax[0].plot(h["d_loss"], "--", label=f"{arch} D")
    if h["fid"]:
        ep, fids, iss = zip(*h["fid"])
        ax[1].plot(ep, fids, marker="o", label=arch)
        ax[2].plot(ep, iss, marker="o", label=arch)
ax[0].set_title("loss"); ax[0].set_xlabel("epoch"); ax[0].legend(fontsize=8)
ax[1].set_title("FID (lower is better)"); ax[1].set_xlabel("epoch"); ax[1].legend()
ax[2].set_title("Inception Score (higher is better)"); ax[2].set_xlabel("epoch"); ax[2].legend()
plt.tight_layout(); plt.show()

# ---- summary table + winner (lowest best-FID) ----
print(f"{'architecture':14}{'best FID':>10}{'final FID':>11}{'final IS':>10}")
winner, winner_fid = None, float("inf")
for arch in ARCHITECTURES:
    h = histories[arch]["fid"]
    if not h:
        continue
    fids = [f for _, f, _ in h]
    best = min(fids)
    print(f"{arch:14}{best:10.2f}{fids[-1]:11.2f}{h[-1][2]:10.2f}")
    if best < winner_fid:
        winner, winner_fid = arch, best
print(f"\nWINNER: {winner}  (best FID {winner_fid:.2f})")

# ---- promote winner to the deployment weight + record it ----
import shutil
shutil.copy(os.path.join(WEIGHTS_DIR, f"{PREFIX}_{winner}_generator_best.pth"),
            os.path.join(WEIGHTS_DIR, f"{PREFIX}_generator.pth"))
wp = os.path.join(WEIGHTS_DIR, "winners.json")
winners = json.load(open(wp)) if os.path.exists(wp) else {}
winners[PREFIX] = winner
json.dump(winners, open(wp, "w"))
print(f"deployment default -> {PREFIX}_generator.pth ({winner}); winners.json updated")

## Best-generator grids, side by side

In [ ]:
fig, axes = plt.subplots(1, len(ARCHITECTURES) + 1, figsize=(4 * (len(ARCHITECTURES) + 1), 4))
axes[0].imshow(make_grid(real_batch[:16], nrow=4, normalize=True).permute(1, 2, 0).squeeze(), cmap=CMAP)
axes[0].set_title("real"); axes[0].axis("off")
for ax, arch in zip(axes[1:], ARCHITECTURES):
    g = Generator(img_size=IMG_SIZE, nz=NZ, channels=CHANNELS).to(device)
    g.load_state_dict(torch.load(os.path.join(WEIGHTS_DIR, f"{PREFIX}_{arch}_generator_best.pth"),
                                 map_location=device))
    g.eval()
    with torch.no_grad():
        samp = g(torch.randn(16, NZ, 1, 1, device=device)).cpu()
    ax.imshow(make_grid(samp, nrow=4, normalize=True).permute(1, 2, 0).squeeze(), cmap=CMAP)
    ax.set_title(f"{arch} (best)"); ax.axis("off")
plt.tight_layout(); plt.savefig(os.path.join(SAMPLES_DIR, f"{PREFIX}_architecture_comparison.png")); plt.show()